# 05 · Evaluation, comparison and responsible-AI analysis

Foundations of Generative AI (ICT4403) — Assignment 3 · English → Hindi

This notebook collects the outputs written by notebooks 01–04 (`results/<model>/`) and computes all metrics on the
**same 1,000-sentence test set**. It needs no GPU.

* **BLEU** (sacreBLEU, `intl` tokenisation, so that the Devanagari danda `।` is split like other punctuation) and **chrF / chrF++**.
  chrF works at the character level, which makes it more reliable than BLEU for a morphologically rich language like Hindi.
* Training / fine-tuning time, inference time (greedy decoding of the 1,000 sentences on one T4) and parameter counts.
* Hallucination and error indicators, the sampling-temperature study, and the bias / sensitive-content probe set.

In [ ]:
import os, json, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sacrebleu
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 250)

ORDER = ["transformer", "mt5", "llama_zeroshot", "llama_lora", "gemma_zeroshot", "gemma_lora"]
SHORT = dict(transformer="Transformer", mt5="mT5-small", llama_zeroshot="Llama zero-shot", llama_lora="Llama LoRA",
             gemma_zeroshot="Gemma zero-shot", gemma_lora="Gemma LoRA")
COLORS = dict(transformer="#4C72B0", mt5="#55A868", llama_zeroshot="#F2B880", llama_lora="#DD8452",
              gemma_zeroshot="#C9A3E6", gemma_lora="#8172B3")
keys = [k for k in ORDER if os.path.exists(f"results/{k}/stats.json")]
read = lambda k, f: pd.read_json(f"results/{k}/{f}", lines=True)
stats = {k: json.load(open(f"results/{k}/stats.json")) for k in keys}
preds = {k: read(k, "test_preds.jsonl") for k in keys}
probes = {k: read(k, "probe_preds.jsonl") for k in keys}
temps = {k: read(k, "temperature_preds.jsonl") for k in keys if os.path.exists(f"results/{k}/temperature_preds.jsonl")}
refs = preds[keys[0]].hi.tolist()
assert all(preds[k].en.tolist() == preds[keys[0]].en.tolist() for k in keys), "models must share the test set"
print("models:", keys, "| test sentences:", len(refs))
os.makedirs("results/eval", exist_ok=True)

## 1. Translation quality and computational cost

In [ ]:
rows = []
for k in keys:
    s, hyp = stats[k], preds[k].hyp.fillna("").tolist()
    rows.append({
        "model": s["model"], "architecture": s["architecture"], "adaptation": s["adaptation"],
        "BLEU": sacrebleu.corpus_bleu(hyp, [refs], tokenize="intl").score,
        "chrF": sacrebleu.corpus_chrf(hyp, [refs]).score,
        "chrF++": sacrebleu.corpus_chrf(hyp, [refs], word_order=2).score,
        "params (M)": s["params"] / 1e6, "trainable (M)": s["trainable_params"] / 1e6,
        "train pairs": s["train_pairs"], "train time (min)": s["train_time_s"] / 60,
        "inference time (s)": s["infer_time_s"], "sentences/s": s["infer_sent_per_s"],
        "HI tokens/sentence": s["avg_tokens_hi"], "EN tokens/sentence": s["avg_tokens_en"],
    })
results = pd.DataFrame(rows, index=keys)
results.to_csv("results/eval/main_results.csv")
results.round(2)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 8))
lab, col = [SHORT[k] for k in keys], [COLORS[k] for k in keys]
for a, c, t in zip(ax.flat, ["BLEU", "chrF", "params (M)", "train time (min)", "inference time (s)", "HI tokens/sentence"],
                   ["BLEU (test)", "chrF (test)", "Parameters (millions, log scale)", "Training / fine-tuning time (min)",
                    "Inference time for 1,000 sentences (s)", "Tokens per Hindi reference sentence"]):
    a.bar(lab, results[c], color=col); a.set_title(t); a.tick_params(axis="x", rotation=30)
    for i, v in enumerate(results[c]): a.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
ax[0, 2].set_yscale("log")
plt.tight_layout(); plt.savefig("results/eval/model_comparison.png", dpi=120); plt.show()

plt.figure(figsize=(6.5, 4.5))
for k in keys:
    plt.scatter(results.loc[k, "params (M)"], results.loc[k, "chrF"], s=120, color=COLORS[k], label=SHORT[k],
                marker="o" if stats[k]["architecture"] == "encoder-decoder" else "s")
plt.xscale("log"); plt.xlabel("parameters (M, log)"); plt.ylabel("chrF"); plt.title("Quality vs. size (o enc-dec, ■ decoder-only)")
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig("results/eval/quality_vs_size.png", dpi=120); plt.show()

## 2. Error and hallucination indicators (test set)

These automatic indicators flag likely failures. Each one is the % of the 1,000 test sentences that it flags:
* **empty**: no output.
* **not Hindi**: under 50 % of the letters are Devanagari (untranslated, copied English or wrong script).
* **over-generation**: the output is more than 2× the reference length (added or hallucinated content, or loops).
* **under-generation**: the output is less than 0.5× the reference length (content dropped).
* **repetition loop**: some word 3-gram occurs 3 or more times.
* **number mismatch**: a number in the source is missing from the output (Devanagari digits count as a match). This catches hallucinated or altered facts.

In [ ]:
DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
def deva_frac(s):
    l = [c for c in s if c.isalpha() or "ऀ" <= c <= "ॿ"]
    return sum("ऀ" <= c <= "ॿ" for c in l) / max(len(l), 1)
def looped(s):
    w = s.split(); g = list(zip(w, w[1:], w[2:]))
    return bool(g) and max(pd.Series(g).value_counts()) >= 3
def number_missing(src, hyp):
    nums = re.findall(r"\d+", src.replace(",", ""))
    h = hyp.translate(DEV_DIGITS).replace(",", "")
    return any(n not in h for n in nums)

def indicators(df):
    hyp, ref = df.hyp.fillna(""), df.hi
    ratio = hyp.str.len() / ref.str.len().clip(lower=1)
    return pd.Series({
        "empty %": (hyp.str.strip() == "").mean() * 100,
        "not Hindi %": (hyp.map(deva_frac) < 0.5).mean() * 100,
        "over-generation %": (ratio > 2).mean() * 100,
        "under-generation %": (ratio < 0.5).mean() * 100,
        "repetition loop %": hyp.map(looped).mean() * 100,
        "number mismatch %": np.mean([number_missing(s, h) for s, h in zip(df.en, hyp)]) * 100,
        "length ratio (hyp/ref)": hyp.str.len().sum() / ref.str.len().sum(),
    })
errors = pd.DataFrame({SHORT[k]: indicators(preds[k]) for k in keys}).T
errors.to_csv("results/eval/error_indicators.csv"); errors.round(1)

In [ ]:
# Side-by-side test translations (first 6 test sentences)
side = pd.DataFrame({"EN": preds[keys[0]].en, "REF": refs, **{SHORT[k]: preds[k].hyp for k in keys}}).head(6)
side.to_csv("results/eval/sample_translations.csv", index=False)
for _, r in side.iterrows():
    print("\n".join(f"{c:16s}: {r[c]}" for c in side.columns), "\n" + "-" * 120)

## 3. Sampling-temperature study

The first 200 test sentences are translated by sampling from the softmax at temperature T ∈ {0.5, 1.0, 1.5}.
Greedy decoding (T → 0, the main results) is included for reference. Measures:
**BLEU / chrF** (adequacy and fluency), **distinct-2** (unique word bigrams ÷ all bigrams over the 200 outputs, a diversity measure),
**repetition** (% of outputs with a word-3-gram loop) and **length ratio**.

In [ ]:
def distinct2(hyps):
    g = [b for h in hyps for b in zip(h.split(), h.split()[1:])]
    return len(set(g)) / max(len(g), 1)

trows = []
for k, df in temps.items():
    greedy = preds[k].iloc[: df[df.temperature == df.temperature.iloc[0]].shape[0]].assign(temperature=0.0)
    for T, g in pd.concat([greedy, df]).groupby("temperature"):
        hyp = g.hyp.fillna("").tolist()
        trows.append(dict(model=SHORT[k], key=k, T=T,
                          BLEU=sacrebleu.corpus_bleu(hyp, [g.hi.tolist()], tokenize="intl").score,
                          chrF=sacrebleu.corpus_chrf(hyp, [g.hi.tolist()]).score,
                          distinct2=distinct2(hyp), repetition=np.mean([looped(h) for h in hyp]) * 100,
                          len_ratio=sum(map(len, hyp)) / g.hi.str.len().sum()))
tstudy = pd.DataFrame(trows)
tstudy.drop(columns="key").to_csv("results/eval/temperature_study.csv", index=False)

fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for k, g in tstudy.groupby("key", sort=False):
    for a, c in zip(ax, ["BLEU", "chrF", "distinct2", "repetition"]):
        a.plot(g["T"], g[c], "o-", color=COLORS[k], label=SHORT[k])
for a, t in zip(ax, ["BLEU", "chrF", "distinct-2 (diversity)", "outputs with repetition loops (%)"]):
    a.set_title(t); a.set_xlabel("temperature (0 = greedy)"); a.set_xticks([0, 0.5, 1.0, 1.5])
ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig("results/eval/temperature_effect.png", dpi=120); plt.show()
tstudy.drop(columns="key").pivot(index="model", columns="T", values=["BLEU", "chrF", "distinct2", "repetition"]).round(2)

In [ ]:
# One sentence at all temperatures, for every model
i = 3
print("EN :", temps[next(iter(temps))].en.iloc[i]); print("REF:", refs[i], "\n")
for k, df in temps.items():
    print(f"{SHORT[k]:12s} greedy: {preds[k].hyp[i]}")
    for T in sorted(df.temperature.unique()):
        print(f"{'':12s} T={T}: {df[df.temperature == T].hyp.iloc[i]}")
    print()

## 4. Responsible-AI probes

### 4.1 Occupational gender bias
Hindi verbs agree with the gender of the subject, so *"The doctor is going home now."* must be rendered as either
जा **रहा** है (masculine) or जा **रही** है (feminine). The English source gives no gender. We detect the gender each
model chose from masculine (रहा/था/गया/थका/हुआ…) and feminine (रही/थी/गई/थकी/हुई…) verb forms.

In [ ]:
FEM = {"रही", "रहीं", "थी", "थीं", "गई", "गयी", "गईं", "थकी", "हुई", "जाती", "करती", "थकीं"}
MASC = {"रहा", "रहे", "था", "थे", "गया", "गए", "गये", "थका", "थके", "हुआ", "हुए", "जाता", "करता"}
def gender(h):
    w = set(re.findall(r"[ऀ-ॣ०-ॿ]+", h or ""))
    f, m = bool(w & FEM), bool(w & MASC)
    return "F" if f and not m else "M" if m and not f else "?"

g = pd.DataFrame({SHORT[k]: probes[k][probes[k].category == "gender"].hyp.map(gender).values for k in keys})
g.insert(0, "sentence", probes[keys[0]][probes[keys[0]].category == "gender"].en.values)
g.to_csv("results/eval/gender_probe.csv", index=False)
display(g)
counts = g.drop(columns="sentence").apply(lambda c: c.value_counts()).fillna(0).astype(int).reindex(["M", "F", "?"]).fillna(0)
counts

In [ ]:
occ = probes[keys[0]][probes[keys[0]].category == "gender"].occupation.values
fem_by_occ = pd.DataFrame({SHORT[k]: (g[SHORT[k]] == "F").values for k in keys}, index=occ).groupby(level=0, sort=False).sum()
plt.figure(figsize=(9, 4)); plt.imshow(fem_by_occ.values, cmap="Purples", vmin=0, vmax=2, aspect="auto")
plt.yticks(range(len(fem_by_occ)), fem_by_occ.index); plt.xticks(range(len(keys)), fem_by_occ.columns, rotation=30)
plt.title("Feminine verb forms chosen (out of 2 sentences per occupation)"); plt.colorbar()
plt.tight_layout(); plt.savefig("results/eval/gender_bias.png", dpi=120); plt.show()

### 4.2 Gender-neutral, cultural/idiomatic, sensitive and hallucination-trap sentences

In [ ]:
other = probes[keys[0]][probes[keys[0]].category != "gender"][["category", "en"]].copy()
for k in keys:
    other[SHORT[k]] = probes[k][probes[k].category != "gender"].hyp.values
other.to_csv("results/eval/probe_translations.csv", index=False)
for _, r in other.iterrows():
    print(f"[{r.category}] {r.en}")
    for k in keys: print(f"   {SHORT[k]:16s}: {r[SHORT[k]]}")
    print()

## 5. Analysis

<!-- ANALYSIS -->